In [1]:
import matplotlib.pyplot as plt
import cv2
import os
import torch
import torchvision
from PIL import Image
import numpy as np
from torch.utils.data import Dataset
from pycocotools.coco import COCO
from pathlib import Path

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cuda


In [3]:
project_dir = Path.cwd().parent
print(project_dir)

c:\Users\feran\Projects\chessify


In [8]:
from torchvision import models, transforms

# class names
label_list = ["", "B", "K", "N", "P", "Q", "R", "b", "k", "n", "p", "q", "r", ""]

# number of classes (including background)
num_classes = len(label_list)

# Load the same model architecture used for training, but without pre-trained weights
model = models.detection.fasterrcnn_resnet50_fpn(weights=False, num_classes=num_classes)

# Load the trained model and set it to evaluation mode
model.load_state_dict(torch.load(project_dir / 'models' / 'model_epoch_10.pth'))
model.eval()

for image_path in (project_dir / "notebooks" / "game_opera").glob("*.jpg"):
    image_bgr = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(image_rgb)

    transform = transforms.Compose([transforms.ToTensor()])
    image_tensor = transform(image_pil).unsqueeze(0)

    # Inference
    with torch.no_grad():
        predictions = model(image_tensor)

    # Detection data
    boxes = predictions[0]['boxes']
    labels = predictions[0]['labels']
    scores = predictions[0]['scores']

    threshold = 0.5
    for i in range(len(boxes)):
        if scores[i] > threshold:
            box = boxes[i].cpu().numpy().astype(int)
            label = label_list[labels[i]]
            score = scores[i].item()

            text = f"{label}: {score:.2f}"
            cv2.putText(image_bgr, text, (box[0], box[1] + 100), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 255, 0), 2, cv2.LINE_AA)

            # Draw bounding box and label
            cv2.rectangle(image_bgr, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)

            name = image_path.name.rstrip('.jpg')
            cv2.imwrite(f"{project_dir}/notebooks/detections/{name}_detections.jpg", image_bgr)
    print(f"Saved detection image: {name}_detections.jpg")

Saved detection image: game_opera0_detections.jpg
Saved detection image: game_opera1_detections.jpg
Saved detection image: game_opera10_detections.jpg
Saved detection image: game_opera11_detections.jpg
Saved detection image: game_opera12_detections.jpg
Saved detection image: game_opera13_detections.jpg
Saved detection image: game_opera14_detections.jpg
Saved detection image: game_opera15_detections.jpg
Saved detection image: game_opera16_detections.jpg
Saved detection image: game_opera17_detections.jpg
Saved detection image: game_opera18_detections.jpg
Saved detection image: game_opera19_detections.jpg
Saved detection image: game_opera2_detections.jpg
Saved detection image: game_opera20_detections.jpg
Saved detection image: game_opera21_detections.jpg
Saved detection image: game_opera22_detections.jpg
Saved detection image: game_opera23_detections.jpg
Saved detection image: game_opera24_detections.jpg
Saved detection image: game_opera25_detections.jpg
Saved detection image: game_opera2